# Image → Text → Summary → Sentiment using Transformers `pipeline()`

## Objective

This notebook performs the same workflow using **Transformers pipelines directly**, without LangChain.

We will use three separate pipeline tasks:

```text
Image
  ↓
image-text-to-text pipeline
  ↓
Extracted text
  ├──→ summarization pipeline
  └──→ sentiment-analysis pipeline
```

This notebook is useful for explaining the difference between:

- using raw Hugging Face Transformers;
- using `langchain_huggingface`;
- using `langchain_openai`.

## Step 1 — Install libraries

We keep Transformers below version 5 in this example to maintain broad compatibility with the pipeline/model combinations demonstrated in the notebook.

In [ ]:
# Uncomment and run once if needed.
# %pip install -U "transformers<5" torch pillow accelerate sentencepiece pandas

## Step 2 — Import libraries

In [ ]:
from pathlib import Path

import pandas as pd
from PIL import Image
from IPython.display import display
from transformers import pipeline

## Step 3 — Load and display the image

In [ ]:
IMAGE_PATH = Path("believe_in_yourself.png")

image = Image.open(IMAGE_PATH).convert("RGB")
print("Image size:", image.size)
display(image)

## Step 4 — Create the image-to-text pipeline

The first pipeline performs the visual task.

A vision-language model receives:

- the image;
- a text instruction.

Its output is generated text.

This can be used for image descriptions and, when prompted appropriately, text extraction.

In [ ]:
VISION_MODEL = "llava-hf/llava-interleave-qwen-0.5b-hf"

image_to_text = pipeline(
    task="image-text-to-text",
    model=VISION_MODEL,
    device=-1
)

print("Vision model:", VISION_MODEL)

## Step 5 — Extract visible text from the image

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {
                "type": "text",
                "text": (
                    "Read all visible text in this image. "
                    "Return the text in its natural reading order. "
                    "Do not summarize or explain."
                ),
            },
        ],
    }
]

ocr_result = image_to_text(
    text=messages,
    max_new_tokens=250,
    return_full_text=False
)

ocr_result

## Step 6 — Clean the pipeline output

In [ ]:
generated = ocr_result[0].get("generated_text", "")

if isinstance(generated, list):
    text_parts = []
    for item in generated:
        if isinstance(item, dict):
            content = item.get("content", "")
            if isinstance(content, str):
                text_parts.append(content)
    extracted_text = "\n".join(text_parts).strip()
else:
    extracted_text = str(generated).strip()

print("EXTRACTED TEXT")
print("=" * 70)
print(extracted_text)

## Step 7 — Create a summarization pipeline

This time we load a model specifically trained for summarization.

The `pipeline()` abstraction automatically handles:

```text
Text
 ↓
Tokenizer
 ↓
Model
 ↓
Generated tokens
 ↓
Decoded summary
```

In [ ]:
SUMMARIZATION_MODEL = "sshleifer/distilbart-cnn-12-6"

summarizer = pipeline(
    task="summarization",
    model=SUMMARIZATION_MODEL,
    device=-1
)

print("Summarization model:", SUMMARIZATION_MODEL)

## Step 8 — Summarize the extracted text

`max_length` controls the maximum generated summary length.

`min_length` helps prevent extremely short output.

`do_sample=False` makes the result deterministic.

In [ ]:
summary_result = summarizer(
    extracted_text,
    max_length=70,
    min_length=15,
    do_sample=False
)

summary = summary_result[0]["summary_text"]

print("SUMMARY")
print("=" * 70)
print(summary)

## Step 9 — Create a sentiment-analysis pipeline

This pipeline uses a model trained specifically for sentiment classification.

Unlike generative prompting, the classifier directly predicts a label and score.

In [ ]:
SENTIMENT_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

sentiment_analyzer = pipeline(
    task="sentiment-analysis",
    model=SENTIMENT_MODEL,
    device=-1
)

print("Sentiment model:", SENTIMENT_MODEL)

## Step 10 — Analyze sentiment

The pipeline returns:

- `label`: predicted sentiment;
- `score`: model confidence.

For this model, the primary labels are `POSITIVE` and `NEGATIVE`.

In [ ]:
sentiment_result = sentiment_analyzer(extracted_text)[0]

print("SENTIMENT")
print("=" * 70)
print("Label:", sentiment_result["label"])
print("Confidence:", round(sentiment_result["score"], 4))

## Step 11 — Combine all outputs

In [ ]:
report = pd.DataFrame([{
    "image": IMAGE_PATH.name,
    "extracted_text": extracted_text,
    "summary": summary,
    "sentiment": sentiment_result["label"],
    "sentiment_confidence": sentiment_result["score"]
}])

report

## Step 12 — Save results

In [ ]:
OUTPUT_FILE = "transformers_pipeline_image_text_analysis.csv"
report.to_csv(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE)

# Comparison of the Three Approaches

| Feature | `langchain_openai` | `langchain_huggingface` | Transformers Pipeline |
|---|---|---|---|
| Image model | OpenAI multimodal model | HF vision model | HF vision model |
| Text processing | OpenAI LLM | Local HF model wrapped by LangChain | Dedicated HF pipelines |
| LangChain used? | Yes | Yes | No |
| API key required? | OpenAI key | Usually no for public local models | Usually no for public local models |
| Local model download | No | Yes | Yes |
| Local RAM/GPU usage | Low | Can be high | Can be high |
| Prompt-based tasks | Yes | Yes | Partly |
| Dedicated classifier | Not required | Not required | Used here |
| Best teaching point | Multimodal LLM workflow | LangChain abstraction over HF | Raw Transformers workflow |

## Main learning

The **business workflow is the same** in all notebooks:

```text
Read image → obtain text → understand text
```

but the implementation changes depending on the framework and model provider.